# Audio Processing : librosa, torchaudio & modeles pre-entraines




**Inspiration concrete** : l'epreuve "Whisper or Shout" de la 3e Olympiade Polonaise d'IA (2026, qualificatif IOAI) demande de classifier des enregistrements de parole selon leur intensite (chuchotement vs cri). On va construire depuis zero un pipeline tres proche de cette tache, pour que la logique soit directement transferable.

Plan :
1. Fondamentaux du signal audio (echantillonnage, formules de base)
2. Charger et visualiser de l'audio (librosa vs torchaudio)
3. Les transformations classiques et leurs formules (STFT, spectrogramme, echelle Mel, MFCC)
4. Pipeline complet de A a Z : classifieur "chuchotement vs cri" (inspire de l'epreuve polonaise)
5. Bonnes pratiques specifiques a l'audio (augmentation, duree variable, fuite de donnees)
6. Modeles pre-entraines audio via Hugging Face (Wav2Vec2/HuBERT, Whisper) et ou se situent Voxtral/Qwen-Audio dans le paysage
7. Section debug : bugs volontaires specifiques a l'audio
8. Exercices finaux

**Note** : toutes les demos utilisent de l'audio **genere en synthese** (sinusoides, bruit) via `numpy`/`scipy` aucun fichier ni telechargement necessaire, tout tourne hors-ligne. C'est volontaire : ca isole la logique du pipeline (qui, elle, s'applique identique a de vrais enregistrements).

**A installer avant la seance** :
```bash
pip install librosa torchaudio soundfile transformers
```

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

try:
    import librosa
    import librosa.display
    LIBROSA_DISPONIBLE = True
except ImportError:
    LIBROSA_DISPONIBLE = False
    print("librosa non installe -> pip install librosa")

try:
    import torchaudio
    TORCHAUDIO_DISPONIBLE = True
except ImportError:
    TORCHAUDIO_DISPONIBLE = False
    print("torchaudio non installe -> pip install torchaudio")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device utilise:", device)
print("librosa:", LIBROSA_DISPONIBLE, "| torchaudio:", TORCHAUDIO_DISPONIBLE)

## 1. Fondamentaux du signal audio

Un son, physiquement, c'est une variation de pression de l'air dans le temps une onde continue. Un ordinateur ne peut stocker que des nombres discrets : il faut donc **echantillonner** cette onde continue.

**Analogie** : c'est comme filmer une scene en mouvement. Une camera ne capture pas un mouvement continu, elle prend des photos successives (les "frames") a intervalles reguliers. Le son numerique, c'est pareil : on "prend une photo" de l'amplitude de l'onde `sr` fois par seconde (`sr` = sample rate, en Hz).

### Fiche technique : les grandeurs de base

| Grandeur | Definition | Valeur typique |
|---|---|---|
| **Sample rate (`sr`)** | nombre d'echantillons captures par seconde | 16 000 Hz (parole), 44 100 Hz (musique/CD) |
| **Bit depth** | precision de chaque echantillon (nb de valeurs possibles) | 16 bits = 65 536 niveaux d'amplitude |
| **Amplitude** | intensite du signal a un instant donne, typiquement dans [-1, 1] apres normalisation | - |
| **Duree** | nb d'echantillons / sample rate | - |

### Fiche technique : le theoreme de Nyquist-Shannon

Pour representer fidelement une frequence `f`, il faut echantillonner a un taux `sr > 2f`. La frequence maximale representable sans perte est donc :

```
f_max = sr / 2
```

**Analogie** : c'est le phenomene de la "roue de chariot" au cinema (wagon-wheel effect) si la camera (l'echantillonnage) est trop lente par rapport a la vitesse de rotation de la roue, celle-ci semble tourner au ralenti ou meme a l'envers. Sous-echantillonner un son cause le meme type de distorsion, appelee **aliasing** : des frequences trop hautes pour le sample rate choisi "replient" et se deguisent en frequences plus basses, fausses.

In [ ]:
# generer un signal audio synthetique : une sinusoide pure (un "la" a 440 Hz)
sr = 16000            # 16 kHz, standard pour la parole
duree = 1.0           # secondes
frequence = 440       # Hz (note "la")

t = np.linspace(0, duree, int(sr * duree), endpoint=False)
signal = 0.5 * np.sin(2 * np.pi * frequence * t)

print("nb d'echantillons:", len(signal))
print("duree calculee:", len(signal) / sr, "secondes")
print("frequence max representable a ce sample rate (Nyquist):", sr / 2, "Hz")

plt.figure(figsize=(9, 3))
plt.plot(t[:400], signal[:400])   # on affiche seulement les premieres millisecondes, sinon illisible
plt.title("Waveform : 25 ms d'une sinusoide a 440 Hz")
plt.xlabel("temps (s)")
plt.ylabel("amplitude")
plt.tight_layout()
plt.show()

## 2. Charger et visualiser de l'audio : librosa vs torchaudio

Deux bibliotheques dominent l'ecosysteme Python pour l'audio :

| | `librosa` | `torchaudio` |
|---|---|---|
| Objectif principal | analyse et visualisation du signal (recherche, prototypage) | integration native a PyTorch (tensors, GPU, pipelines d'entrainement) |
| Format de sortie | tableaux `numpy` | tensors `torch` |
| Points forts | tres riche en features audio classiques (MFCC, chroma, tempo...), grande communaute | rapide, GPU-friendly, s'integre directement dans un `Dataset`/`DataLoader` |
| Usage typique | exploration, comprendre un signal, prototypage rapide | pipeline d'entrainement de production |

**Analogie** : `librosa`, c'est l'etabli d'un luthier qui inspecte le son a la loupe pour comprendre sa structure ; `torchaudio`, c'est la chaine de production qui doit traiter des milliers de sons vite et directement sur GPU pour entrainer un modele. En pratique on utilise souvent les deux : librosa pour explorer/comprendre, torchaudio dans la boucle d'entrainement.

In [ ]:
from scipy.io import wavfile

# on ecrit un vrai fichier .wav sur disque pour manipuler un fichier concret (comme en vrai)
signal_int16 = (signal * 32767).astype(np.int16)
wavfile.write("exemple_sinusoide.wav", sr, signal_int16)
print("fichier ecrit : exemple_sinusoide.wav")

In [ ]:
# chargement avec librosa
if LIBROSA_DISPONIBLE:
    y_librosa, sr_librosa = librosa.load("exemple_sinusoide.wav", sr=None)   # sr=None = garder le sr d'origine
    print("librosa -> type:", type(y_librosa), "| shape:", y_librosa.shape, "| sr:", sr_librosa)

In [ ]:
# chargement avec torchaudio
if TORCHAUDIO_DISPONIBLE:
    waveform, sr_torchaudio = torchaudio.load("exemple_sinusoide.wav")
    print("torchaudio -> type:", type(waveform), "| shape:", waveform.shape, "| sr:", sr_torchaudio)
    # NOTE : torchaudio retourne une shape (nb_canaux, nb_echantillons) -> ici (1, 16000) car mono
    # librosa retourne un simple vecteur 1D pour du mono

### Fiche technique : mono vs stereo, et le piege du `sr`

- Un fichier **mono** a un seul canal, **stereo** en a deux (gauche/droite). `torchaudio.load` garde toujours la dimension des canaux `(nb_canaux, nb_echantillons)`, meme en mono (`nb_canaux=1`) reflexe à prendre : verifier `.shape` avant de nourrir un modele.
- **Piege frequent** : un modele pre-entraine est toujours entraine à un `sr` precis (souvent 16 000 Hz pour la parole). Si on lui donne de l'audio a un autre sample rate sans reechantillonner, les resultats sont silencieusement faux (le modele "entend" un son deforme, sans lever d'erreur). Toujours verifier et aligner le `sr` avec `torchaudio.transforms.Resample` ou `librosa.resample`.

In [ ]:
# demonstration du resampling (tres important avant d'utiliser un modele pre-entraine)
if TORCHAUDIO_DISPONIBLE:
    nouveau_sr = 8000
    resampler = torchaudio.transforms.Resample(orig_freq=sr_torchaudio, new_freq=nouveau_sr)
    waveform_resample = resampler(waveform)
    print("shape avant resampling:", waveform.shape, "a", sr_torchaudio, "Hz")
    print("shape apres resampling:", waveform_resample.shape, "a", nouveau_sr, "Hz")
    # on a bien moins d'echantillons, mais la duree en secondes reste la meme

## 3. Les transformations classiques et leurs formules

La waveform brute est rarement utilisee directement par un modele : elle est trop longue, trop bruitee, et ne montre pas explicitement les frequences presentes. On transforme le signal pour en extraire des **features** plus informatives.

### 3.1 La transformee de Fourier a court terme (STFT)

La waveform montre l'amplitude au cours du temps, mais pas quelles frequences sont presentes a quel moment. La **Short-Time Fourier Transform** decoupe le signal en petites fenetres qui se chevauchent, et calcule la transformee de Fourier de chaque fenetre :

```
X(m, k) = somme_n  x[n] * w[n - m*H] * exp(-j * 2*pi * k * n / N)
```

- `w` : la fenetre (ex: Hann), de taille `n_fft`
- `H` : le hop length, le decalage entre deux fenetres consecutives (permet le chevauchement)
- `m` : l'indice de la fenetre (l'axe temps du resultat)
- `k` : l'indice de frequence (l'axe frequence du resultat)

**Analogie** : c'est comme lire une partition de musique en la regardant par une petite fenetre glissante qui avance mesure par mesure à chaque position, on note quelles notes (frequences) sont jouees, avec un leger chevauchement pour ne rien manquer entre deux fenetres.

Le **spectrogramme**, c'est simplement le carre (ou le module) de la STFT, affiche comme une image 2D : temps en abscisse, frequence en ordonnee, intensite (souvent en decibels) representee par la couleur.

In [ ]:
if LIBROSA_DISPONIBLE:
    n_fft = 512      # taille de la fenetre d'analyse
    hop_length = 128  # decalage entre deux fenetres

    stft = librosa.stft(y_librosa, n_fft=n_fft, hop_length=hop_length)
    spectrogramme_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)

    plt.figure(figsize=(8, 4))
    librosa.display.specshow(spectrogramme_db, sr=sr_librosa, hop_length=hop_length,
                              x_axis="time", y_axis="hz", cmap="magma")
    plt.colorbar(format="%+2.0f dB")
    plt.title("Spectrogramme (STFT) d'une sinusoide a 440 Hz")
    plt.tight_layout()
    plt.show()
    # on doit voir une bande horizontale nette autour de 440 Hz : la sinusoide est une frequence pure

### 3.2 L'echelle Mel

L'oreille humaine ne percoit pas les frequences de facon lineaire : la difference entre 100 Hz et 200 Hz parait plus grande que celle entre 10 000 Hz et 10 100 Hz, bien que l'ecart en Hz soit identique. L'**echelle Mel** reechelonne les frequences pour se rapprocher de la perception humaine :

```
m = 2595 * log10(1 + f / 700)
```

**Analogie** : c'est comparable a l'echelle de Richter pour les seismes, ou l'echelle des decibels pour le son on compresse une grandeur physique tres etalee (ici, les Hz) pour qu'elle corresponde mieux a une perception humaine qui, elle, fonctionne plutot de facon logarithmique.

Le **mel-spectrogramme** est un spectrogramme dont l'axe des frequences a ete reprojete sur cette echelle Mel (via un banc de filtres triangulaires) c'est la representation la plus utilisee en entree des modeles audio modernes.

In [ ]:
if LIBROSA_DISPONIBLE:
    mel_spec = librosa.feature.melspectrogram(y=y_librosa, sr=sr_librosa, n_fft=n_fft,
                                                hop_length=hop_length, n_mels=64)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

    plt.figure(figsize=(8, 4))
    librosa.display.specshow(mel_spec_db, sr=sr_librosa, hop_length=hop_length,
                              x_axis="time", y_axis="mel", cmap="magma")
    plt.colorbar(format="%+2.0f dB")
    plt.title("Mel-spectrogramme")
    plt.tight_layout()
    plt.show()

### 3.3 Les MFCC (Mel-Frequency Cepstral Coefficients)

Les MFCC vont un cran plus loin : on prend le logarithme du mel-spectrogramme, puis on lui applique une transformee en cosinus discrete (DCT), qui decorrelle les coefficients et resume le signal en un petit nombre de valeurs (souvent 13 a 40) qui capturent l'essentiel du **timbre** du son, independamment de sa hauteur exacte.

**Analogie** : si le mel-spectrogramme est une photo d'identite en tres haute resolution, les MFCC sont le resume signaletique qu'un agent retiendrait ("visage ovale, sourcils epais, nez droit...") une poignee de traits caracteristiques suffisants pour reconnaitre/classer, sans avoir besoin de tous les pixels.

In [ ]:
if LIBROSA_DISPONIBLE:
    mfcc = librosa.feature.mfcc(y=y_librosa, sr=sr_librosa, n_mfcc=13, n_fft=n_fft, hop_length=hop_length)
    print("shape des MFCC:", mfcc.shape)   # (n_mfcc, nb_fenetres_temporelles)

    plt.figure(figsize=(8, 3))
    librosa.display.specshow(mfcc, sr=sr_librosa, hop_length=hop_length, x_axis="time", cmap="viridis")
    plt.colorbar()
    plt.title("MFCC (13 coefficients)")
    plt.tight_layout()
    plt.show()

### Fiche technique : les equivalents dans `torchaudio`

Tout ce qu'on vient de faire avec librosa a un equivalent direct en `torchaudio.transforms`, sous forme de modules PyTorch (utilisables directement dans un `Dataset` ou meme sur GPU) :

| librosa | torchaudio |
|---|---|
| `librosa.stft` | `torchaudio.transforms.Spectrogram(n_fft=..., hop_length=...)` |
| `librosa.feature.melspectrogram` | `torchaudio.transforms.MelSpectrogram(sample_rate=..., n_fft=..., n_mels=...)` |
| `librosa.feature.mfcc` | `torchaudio.transforms.MFCC(sample_rate=..., n_mfcc=...)` |
| `librosa.amplitude_to_db` / `power_to_db` | `torchaudio.transforms.AmplitudeToDB()` |

In [ ]:
if TORCHAUDIO_DISPONIBLE:
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=sr_torchaudio, n_fft=512, hop_length=128, n_mels=64
    )
    mel_spec_torch = mel_transform(waveform)   # directement sur un tensor, utilisable dans une boucle d'entrainement
    print("shape du mel-spectrogramme torchaudio:", mel_spec_torch.shape)
    # (nb_canaux, n_mels, nb_fenetres_temporelles) -> pret a etre nourri a un CNN comme une "image" 

## 4. Pipeline complet de A a Z : classifieur "chuchotement vs cri"

On construit maintenant un pipeline complet, inspire directement de l'epreuve **"Whisper or Shout"** (Olympiade Polonaise d'IA 2026) : classifier un enregistrement de parole selon son intensite.

Pour rester reproductible sans dataset externe, on **simule** des "chuchotements" (faible amplitude, plus de bruit relatif) et des "cris" (forte amplitude, spectre plus riche en harmoniques) la logique du pipeline (features -> Dataset -> modele -> entrainement -> evaluation) est strictement identique a celle qu'on appliquerait sur de vrais enregistrements.

**Analogie du pipeline complet** : c'est exactement le meme "restaurant" que dans le notebook precedent (donnees -> Dataset/DataLoader -> modele -> entrainement -> evaluation -> sauvegarde), seul le type d'ingredient change : ici, des ondes sonores plutot que des pixels.

### 4.1 Generer le dataset synthetique

In [ ]:
def generer_parole_simulee(intensite="chuchotement", duree=1.0, sr=16000, seed=None):
    """Simule un signal de parole en superposant plusieurs harmoniques + bruit,
    avec une amplitude et une richesse spectrale qui dependent de l'intensite."""
    rng = np.random.default_rng(seed)
    t = np.linspace(0, duree, int(sr * duree), endpoint=False)

    if intensite == "chuchotement":
        amplitude_base = rng.uniform(0.02, 0.08)
        nb_harmoniques = rng.integers(1, 3)
        niveau_bruit = 0.03
    else:  # "cri"
        amplitude_base = rng.uniform(0.5, 0.9)
        nb_harmoniques = rng.integers(3, 6)
        niveau_bruit = 0.01

    f0 = rng.uniform(150, 300)   # frequence fondamentale approximative d'une voix
    signal = np.zeros_like(t)
    for k in range(1, nb_harmoniques + 1):
        signal += (amplitude_base / k) * np.sin(2 * np.pi * f0 * k * t + rng.uniform(0, 2 * np.pi))

    bruit = rng.normal(0, niveau_bruit, size=t.shape)
    signal = signal + bruit
    return signal.astype(np.float32)


# apercu rapide
exemple_chuchotement = generer_parole_simulee("chuchotement", seed=0)
exemple_cri = generer_parole_simulee("cri", seed=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(exemple_chuchotement[:2000])
axes[0].set_title("Chuchotement (extrait)")
axes[1].plot(exemple_cri[:2000])
axes[1].set_title("Cri (extrait)")
for ax in axes:
    ax.set_ylim(-1, 1)
plt.tight_layout()
plt.show()

In [ ]:
# construction du dataset complet
torch.manual_seed(0)
np.random.seed(0)

N_PAR_CLASSE = 150
donnees_brutes = []
labels = []

for i in range(N_PAR_CLASSE):
    donnees_brutes.append(generer_parole_simulee("chuchotement", seed=i))
    labels.append(0)   # 0 = chuchotement
    donnees_brutes.append(generer_parole_simulee("cri", seed=i + 10000))
    labels.append(1)   # 1 = cri

print("taille du dataset:", len(donnees_brutes), "exemples")

### 4.2 Extraction de features et `Dataset`/`DataLoader`

Plutot que de nourrir la waveform brute (16 000 valeurs par seconde, tres redondant), on extrait un **mel-spectrogramme** pour chaque exemple : c'est plus compact et deja structure comme une "image" temps/frequence, ideale pour un petit CNN.

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split

class DatasetAudio(Dataset):
    def __init__(self, signaux, labels, sr=16000, n_fft=512, hop_length=128, n_mels=32):
        self.signaux = signaux
        self.labels = labels
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.signaux)

    def __getitem__(self, idx):
        signal = torch.tensor(self.signaux[idx]).unsqueeze(0)   # (1, nb_echantillons) -> mono
        mel = self.mel_transform(signal)
        mel_db = self.db_transform(mel)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mel_db, label


dataset_complet = DatasetAudio(donnees_brutes, labels)

# verification rapide de shape
exemple_mel, exemple_label = dataset_complet[0]
print("shape d'un mel-spectrogramme:", exemple_mel.shape, "| label:", exemple_label.item())

n_val = int(0.2 * len(dataset_complet))
n_train = len(dataset_complet) - n_val
train_dataset, val_dataset = random_split(dataset_complet, [n_train, n_val],
                                           generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
print("batches d'entrainement:", len(train_loader))

### 4.3 Definir un petit CNN audio (le mel-spectrogramme est traite comme une image)

In [ ]:
import torch.nn as nn

class PetitCNNAudio(nn.Module):
    def __init__(self, nb_classes=2):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()
        self.pool_adapt = nn.AdaptiveAvgPool2d((4, 4))   # rend le modele robuste a des durees d'entree variables
        self.fc = nn.Linear(16 * 4 * 4, nb_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool_adapt(x)
        x = x.flatten(1)
        return self.fc(x)


modele_audio = PetitCNNAudio().to(device)
batch_exemple, _ = next(iter(train_loader))
sortie = modele_audio(batch_exemple.to(device))
print("shape de sortie:", sortie.shape)  # (batch, 2)

**Fiche technique : `AdaptiveAvgPool2d`**

Contrairement a `MaxPool2d(kernel_size=2)` qui divise toujours la taille par un facteur fixe, `AdaptiveAvgPool2d((h, w))` **force** la sortie a une taille fixe `(h, w)`, quelle que soit la taille d'entree. Tres utile en audio : deux enregistrements de duree differente donnent des mel-spectrogrammes de largeur differente `AdaptiveAvgPool2d` permet quand meme de les nourrir au meme `nn.Linear` final, sans avoir a tronquer/rembourrer tout au meme nombre exact d'echantillons en amont.

### 4.4 Entrainement et evaluation

In [ ]:
import torch.optim as optim

def evaluer_audio(modele, loader):
    modele.eval()
    correct, total, perte_totale = 0, 0, 0.0
    fonction_loss = nn.CrossEntropyLoss()
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            sortie = modele(x_batch)
            perte_totale += fonction_loss(sortie, y_batch).item() * x_batch.size(0)
            correct += (sortie.argmax(dim=1) == y_batch).sum().item()
            total += y_batch.size(0)
    modele.train()
    return perte_totale / total, correct / total


fonction_loss = nn.CrossEntropyLoss()
optimizer = optim.Adam(modele_audio.parameters(), lr=1e-3)

nb_epochs = 10
for epoch in range(nb_epochs):
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        sortie = modele_audio(x_batch)
        loss = fonction_loss(sortie, y_batch)
        loss.backward()
        optimizer.step()

    val_loss, val_acc = evaluer_audio(modele_audio, val_loader)
    if epoch % 2 == 0:
        print(f"epoch {epoch+1}/{nb_epochs} | val_loss {val_loss:.4f} | val_acc {val_acc:.4f}")

print("\nTache tres simple ici (amplitude tres differente entre les 2 classes) -> "
      "la precision devrait rapidement approcher 100%. Sur de vrais enregistrements, "
      "attends-toi a un probleme bien plus difficile.")

## 5. Bonnes pratiques specifiques a l'audio

### 5.1 Duree variable : padding, troncature, et `collate_fn`
Contrairement aux images (taille fixe apres resize), l'audio a naturellement des durees variables. Trois strategies, a combiner selon le cas :
- **Troncature/padding a une duree fixe** avant transformation (ex: toujours forcer 3 secondes, remplir de zeros si plus court, couper si plus long)
- **`AdaptiveAvgPool2d`** en fin de reseau (vu ci-dessus), pour tolerer des tailles de spectrogramme differentes
- **`collate_fn` personnalise** dans le `DataLoader`, qui rembourre chaque batch a la longueur du plus long element du batch (plus efficace que rembourrer tout le dataset a la longueur max globale)

### 5.2 Augmentation de donnees audio
Comme pour les images, on peut augmenter artificiellement la diversite du dataset d'entrainement :
- **Ajout de bruit** (bruit blanc, bruit ambiant) rend le modele robuste aux conditions d'enregistrement reelles
- **Time stretching** (accelerer/ralentir sans changer la hauteur) et **pitch shifting** (changer la hauteur sans changer la vitesse) `librosa.effects.time_stretch` / `librosa.effects.pitch_shift`
- **SpecAugment** : masquer aleatoirement des bandes de temps ou de frequence directement sur le spectrogramme (plutot que sur la waveform) très utilise en pratique, simple et efficace

### 5.3 Le piege n1 : la fuite de donnees (data leakage) par locuteur
Si plusieurs enregistrements viennent du **meme locuteur**, un split train/test aleatoire "au niveau des segments" peut placer des extraits du meme locuteur dans le train ET le test. Le modele apprend alors a reconnaitre la voix plutot qu'a resoudre la vraie tache la precision sur le test parait excellente mais s'effondre sur de nouveaux locuteurs.

**Reflexe a avoir** : toujours splitter par **locuteur/session** (tous les enregistrements d'une meme personne dans un seul cote du split), jamais par segment individuel, des que plusieurs enregistrements partagent une source commune.

### 5.4 Normalisation coherente
Le sample rate, mais aussi l'amplitude (normaliser entre -1 et 1, ou par la racine carree de l'energie moyenne — le "RMS") doivent etre traites de la **meme facon** a l'entrainement et a l'inference. Un modele entraine sur de l'audio normalise en RMS et teste sur de l'audio brut non normalise degradera silencieusement.

## 6. Modeles audio pre-entraines (Hugging Face)

Comme pour le texte et l'image, on part rarement de zero pour des taches audio complexes : le syllabus IOAI mentionne explicitement des **encodeurs audio pre-entraines** (HuBERT) et des **modeles audio** plus recents (Whisper, Voxtral, Qwen-Audio).

| Modele | Type | Usage principal |
|---|---|---|
| **HuBERT** / **Wav2Vec2** | encodeur audio auto-supervise | transforme l'audio brut en embeddings riches, reutilisables pour classification, reconnaissance de locuteur, etc. (comme BERT mais pour l'audio) |
| **Whisper** (OpenAI) | modele encodeur-decodeur | reconnaissance vocale (speech-to-text) multilingue, robuste au bruit |
| **Voxtral** (Mistral AI) | modele audio-langage | comprehension audio + generation de texte (question-answering sur de l'audio, transcription, etc.) |
| **Qwen-Audio** (Alibaba) | modele audio-langage multimodal | comprehension audio generaliste (parole, sons, musique) couplee a un modele de langage |

**Analogie** : HuBERT/Wav2Vec2, c'est un traducteur qui convertit le son en une representation abstraite mais tres riche ("langue interne" du modele) ; Whisper, c'est un secretaire qui ecoute et transcrit ; Voxtral/Qwen-Audio, ce sont des assistants qui a la fois ecoutent ET discutent intelligemment de ce qu'ils ont entendu.

In [ ]:
try:
    from transformers import pipeline
    HF_DISPONIBLE = True
except ImportError:
    HF_DISPONIBLE = False
    print("transformers non installe -> pip install transformers")

if HF_DISPONIBLE:
    # reconnaissance vocale avec Whisper (petite version, rapide)
    # NOTE : sur notre sinusoide synthetique, Whisper ne "reconnaitra" rien de coherent
    #        (ce n'est pas de la vraie parole) -- ceci montre juste le pattern d'utilisation.
    transcripteur = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")
    resultat = transcripteur("exemple_sinusoide.wav")
    print(resultat)

In [ ]:
if HF_DISPONIBLE:
    from transformers import AutoFeatureExtractor, AutoModel

    nom_modele = "facebook/hubert-base-ls960"
    extracteur = AutoFeatureExtractor.from_pretrained(nom_modele)
    modele_hubert = AutoModel.from_pretrained(nom_modele)

    entree = extracteur(y_librosa, sampling_rate=sr_librosa, return_tensors="pt")
    with torch.no_grad():
        sortie_hubert = modele_hubert(**entree)

    print("shape des embeddings HuBERT:", sortie_hubert.last_hidden_state.shape)
    # (1, nb_frames, taille_embedding) : un vecteur riche par petite tranche de temps,
    # utilisable ensuite comme feature d'entree pour n'importe quel classifieur

**Aller plus loin (semaine)** : au lieu d'entrainer un CNN from scratch sur des mel-spectrogrammes (section 4), on peut extraire des embeddings HuBERT/Wav2Vec2 pre-entraines et n'entrainer qu'une petite tete de classification par-dessus (fine-tuning partiel) souvent bien plus performant avec peu de donnees, exactement le meme principe que le fine-tuning vu sur le texte dans le notebook precedent.

## 7. Section debug : bugs specifiques a l'audio

Meme methode que dans le notebook precedent : chaque cellule contient un bug volontaire. Lis attentivement les messages d'erreur (ou l'absence d'erreur mais un resultat absurde) avant de corriger.

### Bug audio 1 : mismatch de sample rate

Ce code est cense reechantillonner un signal de 16000 Hz vers 8000 Hz avant de le donner a un modele qui attend du 8000 Hz. Le resultat plante ou est incoherent.

In [ ]:
# BUG AUDIO 1 (a corriger)
if TORCHAUDIO_DISPONIBLE:
    signal_original, sr_original = torchaudio.load("exemple_sinusoide.wav")  # sr_original = 16000

    resampler_bug = torchaudio.transforms.Resample(orig_freq=8000, new_freq=8000)  # <- regarde bien orig_freq
    signal_resample_bug = resampler_bug(signal_original)

    print("sr d'origine reel:", sr_original)
    print("shape apres 'resampling':", signal_resample_bug.shape)
    # indice : Resample a besoin de connaitre le VRAI sample rate d'origine pour bien fonctionner

*Corrige le bug ici :*

In [ ]:
# ta version corrigee



### Bug audio 2 : stereo non gere

Ce code calcule un mel-spectrogramme, cense fonctionner sur n'importe quel fichier, mais plante des qu'on lui donne un fichier stereo (2 canaux) au lieu d'un fichier mono.

In [ ]:
# BUG AUDIO 2 (a corriger)
if TORCHAUDIO_DISPONIBLE:
    signal_stereo = torch.randn(2, 16000)   # simule un fichier stereo : shape (2, nb_echantillons)

    mel_transform_bug = torchaudio.transforms.MelSpectrogram(sample_rate=16000, n_mels=32)
    mel_bug = mel_transform_bug(signal_stereo)

    modele_test = PetitCNNAudio()
    sortie_bug = modele_test(mel_bug)   # plante : le modele attend 1 canal, pas 2
    print(sortie_bug.shape)
    # indice : il faut ramener le stereo a du mono AVANT de calculer le mel-spectrogramme
    # (ex: moyenne des deux canaux, torch.mean(x, dim=0, keepdim=True))

*Corrige le bug ici :*

In [ ]:
# ta version corrigee



### Bug audio 3 : `amplitude_to_db` sur un signal deja en dB

Ce code applique une conversion en decibels une seconde fois par erreur, ce qui rend les valeurs completement incoherentes (des dB de dB n'ont pas de sens physique).

In [ ]:
# BUG AUDIO 3 (a corriger)
if LIBROSA_DISPONIBLE:
    stft_bug = librosa.stft(y_librosa, n_fft=512, hop_length=128)
    spectrogramme_db_bug = librosa.amplitude_to_db(np.abs(stft_bug), ref=np.max)
    spectrogramme_db_bug_bis = librosa.amplitude_to_db(spectrogramme_db_bug, ref=np.max)  # <- probleme ici

    print("min/max apres 1re conversion:", spectrogramme_db_bug.min(), spectrogramme_db_bug.max())
    print("min/max apres 2e conversion:", spectrogramme_db_bug_bis.min(), spectrogramme_db_bug_bis.max())
    # indice : combien de fois doit-on convertir un signal en dB ?

*Corrige le bug ici :*

In [ ]:
# ta version corrigee



## 8. Exercices finaux

**Exercice A — enrichir le classifieur (20-30 min)**
Reprends le pipeline de la section 4. Ajoute une 3e classe `"voix normale"` (amplitude et richesse harmonique intermediaires entre chuchotement et cri) a `generer_parole_simulee`, regenere le dataset avec 3 classes, adapte `nb_classes` du modele, et reentraine. Affiche la matrice de confusion finale (indice : `sklearn.metrics.confusion_matrix` ou calcul manuel avec des tensors).

**Exercice B — augmentation de donnees (15-20 min)**
Ajoute une fonction `augmenter(signal)` qui, avec une probabilite de 50%, ajoute du bruit gaussien supplementaire au signal. Applique-la dans `DatasetAudio.__getitem__` (uniquement sur le split d'entrainement, jamais sur la validation) et verifie que l'entrainement reste stable.

**Exercice C — Hugging Face (15 min)**
Utilise `pipeline("audio-classification")` avec un modele public de classification audio (par exemple un modele de detection de langue ou d'emotion) sur `exemple_sinusoide.wav`. Affiche les labels et scores retournes, meme si le resultat n'a pas de sens sur une sinusoide pure l'objectif est de maitriser le pattern d'appel.

**Exercice D — chrono, inspire de "Whisper or Shout" (25-30 min, sans aide)**
En condition d'epreuve : reconstruis entierement le pipeline de la section 4 (generation -> Dataset -> modele -> entrainement -> evaluation) mais en utilisant des **MFCC** (`torchaudio.transforms.MFCC`) au lieu de mel-spectrogrammes comme feature d'entree. Compare la precision finale a celle obtenue avec les mel-spectrogrammes.

In [ ]:
# Exercice D : ta solution ici



